# Método de previsão com média móvel e métrica MAE

Neste notebook, é construído um baseline usando média móvel de 7 dias para previsão diária de vendas em Janeiro de 2024, considerando o período de treino até 31/12/2023 e teste em janeiro de 2024.

## Leitura da base de dados e conversão de datas

In [1]:
import os
import pandas as pd
from sklearn.metrics import mean_absolute_error

# Ler o CSV de vendas e produtos 
vendas_path = '../datasets/vendas_2023_2024.csv'
if not os.path.exists(vendas_path):
    fallback_path = '../features/vendas_normalizado.csv'
    if os.path.exists(fallback_path):
        vendas_path = fallback_path
    else:
        raise FileNotFoundError('Nenhum arquivo de vendas encontrado em datasets/ ou features/')

vendas = pd.read_csv(vendas_path)
produtos = pd.read_csv('../datasets/produtos_raw.csv')

# Conversão única e robusta de datas (aceita formatos mistos)
vendas['sale_date'] = pd.to_datetime(vendas['sale_date'], errors='coerce', format='mixed', dayfirst=True)
na_dates = vendas['sale_date'].isna().sum()
if na_dates > 0:
    print(f'AVISO: {na_dates} linhas com sale_date inválida serão descartadas')
    vendas = vendas.dropna(subset=['sale_date'])

print('Arquivo de vendas usado:', vendas_path)
print('Período no arquivo:', vendas['sale_date'].min(), 'a', vendas['sale_date'].max())

Arquivo de vendas usado: ../datasets/vendas_2023_2024.csv
Período no arquivo: 2023-01-01 00:00:00 a 2024-12-31 00:00:00


## Mapeamento e filtragem de vendas

In [2]:
# Produto alvo
produto_alvo = 'Motor de Popa Yamaha Evo Dash 155HP'

# Mapeamento para o código do produto (id_product/code)
prod_info = produtos[produtos['name'].str.strip().str.lower() == produto_alvo.strip().lower()]
if prod_info.empty:
    raise ValueError(f"Produto '{produto_alvo}' não encontrado no arquivo produtos_raw.csv")
prod_code = prod_info['code'].iloc[0]
print(f'Código do produto: {prod_code}')

# Filtrando vendas pelo id_product
if 'id_product' not in vendas.columns:
    raise ValueError('Coluna id_product não encontrada em vendas_2023_2024.csv')

df_prod = vendas[vendas['id_product'] == prod_code].copy()
if df_prod.empty:
    raise ValueError(f"Produto '{produto_alvo}' (code={prod_code}) não encontrado em vendas_2023_2024.csv")

print('Linhas produto:', len(df_prod))

Código do produto: 54
Linhas produto: 62


## Etapas de treino e teste

In [3]:
# Vendas diárias pela soma da quantidade
vendas_diarias = df_prod.groupby(df_prod['sale_date'].dt.date).agg({'qtd':'sum'}).rename(columns={'qtd':'vendas'}).reset_index()
vendas_diarias['sale_date'] = pd.to_datetime(vendas_diarias['sale_date'])

# Calendário contínuo para evitar dias ausentes
full_calendar = pd.DataFrame({'sale_date': pd.date_range(vendas_diarias['sale_date'].min(), vendas_diarias['sale_date'].max(), freq='D')})
vendas_diarias = full_calendar.merge(vendas_diarias, on='sale_date', how='left').fillna(0)

# Definindo treino/teste
train_end = pd.Timestamp('2023-12-31')
test_start = pd.Timestamp('2024-01-01')
test_end = pd.Timestamp('2024-01-31')

data_train = vendas_diarias[vendas_diarias['sale_date'] <= train_end].copy()
data_test = vendas_diarias[(vendas_diarias['sale_date'] >= test_start) & (vendas_diarias['sale_date'] <= test_end)].copy()

print('Período treino:', data_train['sale_date'].min(), 'a', data_train['sale_date'].max())
print('Período teste:', data_test['sale_date'].min(), 'a', data_test['sale_date'].max())

Período treino: 2023-01-10 00:00:00 a 2023-12-31 00:00:00
Período teste: 2024-01-01 00:00:00 a 2024-01-31 00:00:00


## Aplicação de previsões e métrica MAE (Erro Absoluto Médio)

In [4]:
# Pegando a sequência completa de 2023-01-01 até 2024-01-31 para o teste
serie = vendas_diarias.set_index('sale_date')['vendas'].rename('vendas').sort_index()

# Criando previsões para todo conjunto usando rolling(7) antes de cada dia
rolling_mean = serie.rolling(window=7, min_periods=1).mean().shift(1)

pred = rolling_mean[test_start:test_end]
real = serie[test_start:test_end]

# Métrica MAE
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(real, pred)

print(f'MAE para janeiro de 2024: {mae:.4f}')

# Resumo resultados
res = pd.DataFrame({'date': pred.index, 'pred': pred.values, 'real': real.values, 'abs_err': (real-pred).abs()})
res.head(10)

MAE para janeiro de 2024: 0.9954


,date,pred,real,abs_err
sale_date,,,,
2024-01-01,2024-01-01,0.0,0.0,0.0
2024-01-02,2024-01-02,0.0,0.0,0.0
2024-01-03,2024-01-03,0.0,0.0,0.0
2024-01-04,2024-01-04,0.0,0.0,0.0
2024-01-05,2024-01-05,0.0,0.0,0.0
2024-01-06,2024-01-06,0.0,0.0,0.0
2024-01-07,2024-01-07,0.0,0.0,0.0
2024-01-08,2024-01-08,0.0,0.0,0.0
2024-01-09,2024-01-09,0.0,0.0,0.0


## Gerando um baseline aprimorado

In [5]:
# Baseline aprimorado: média por dia da semana (treino) + fallback média móvel 7d
from sklearn.metrics import mean_absolute_error

serie_full = vendas_diarias.set_index('sale_date')['vendas'].rename('vendas').sort_index()

# Separação treino/teste na série completa
serie_train = serie_full[:train_end]
serie_test = serie_full[test_start:test_end]

# 1) Média por dia da semana calculada somente no treino
weekday_mean = serie_train.groupby(serie_train.index.dayofweek).mean()
pred_weekday = pd.Series(index=serie_test.index, dtype=float)
pred_weekday[:] = [weekday_mean.get(d, float('nan')) for d in serie_test.index.dayofweek]

# 2) Fallback: média móvel de 7 dias (sem vazamento)
rolling_7 = serie_full.rolling(window=7, min_periods=1).mean().shift(1)
pred_fallback = rolling_7[test_start:test_end]

# Combinação: usa previsão sazonal quando existir, senão fallback
pred_melhor = pred_weekday.fillna(pred_fallback).fillna(0)

mae_base_atual = mean_absolute_error(serie_test, rolling_7[test_start:test_end])
mae_melhor = mean_absolute_error(serie_test, pred_melhor)

print(f'MAE baseline atual (média móvel 7d): {mae_base_atual:.4f}')
print(f'MAE baseline aprimorado (dia da semana + fallback): {mae_melhor:.4f}')

comparativo = pd.DataFrame({
    'real': serie_test,
    'pred_base_7d': rolling_7[test_start:test_end],
    'pred_melhor': pred_melhor,
    'erro_abs_base_7d': (serie_test - rolling_7[test_start:test_end]).abs(),
    'erro_abs_melhor': (serie_test - pred_melhor).abs()
})

comparativo.head(12)

MAE baseline atual (média móvel 7d): 0.9954
MAE baseline aprimorado (dia da semana + fallback): 1.1459


,real,pred_base_7d,pred_melhor,erro_abs_base_7d,erro_abs_melhor
sale_date,,,,,
2024-01-01,0.0,0.0,0.900000,0.0,0.900000
2024-01-02,0.0,0.0,0.705882,0.0,0.705882
2024-01-03,0.0,0.0,0.725490,0.0,0.725490
2024-01-04,0.0,0.0,0.588235,0.0,0.588235
2024-01-05,0.0,0.0,0.686275,0.0,0.686275
2024-01-06,0.0,0.0,0.529412,0.0,0.529412
2024-01-07,0.0,0.0,0.725490,0.0,0.725490
2024-01-08,0.0,0.0,0.900000,0.0,0.900000
2024-01-09,0.0,0.0,0.705882,0.0,0.705882
